## 13. Summary and Next Steps

### What We've Built:
1. **Multi-task Learning Model**: EfficientNetB3 with classification and regression heads
2. **Food Classification**: 101 food classes from Food-101 dataset
3. **Calorie Estimation**: Both lookup-based and regression-based approaches
4. **Production-Ready Inference**: Exported models in multiple formats
5. **API Integration Ready**: Models saved and ready for Flask/Streamlit integration

### Model Outputs:
- Trained model weights
- Calorie lookup database (JSON/CSV)
- Class labels mapping
- Model configuration

### Usage in Production:
```python
# Inference example
predictions = predict_food(image_path, model, class_labels, calorie_lookup)
for pred in predictions['predictions']:
    print(f"{pred['class']}: {pred['confidence']}% - {pred['estimated_calories']} cal")
```

### Next Steps:
1. Deploy model to Flask API (`src/api.py`)
2. Build Streamlit UI (`ui/app.py`)
3. Integrate with calorie tracker (`src/inference.py`)
4. Add user authentication and database
5. Deploy to cloud (AWS/GCP/Azure)

In [1]:
def predict_food(image_path, model, class_labels, calorie_lookup, top_k=5):
    """
    Predict food class and calorie content from image.
    
    Args:
        image_path: Path to image file
        model: Trained model
        class_labels: List of class names
        calorie_lookup: Dictionary mapping class to calories
        top_k: Number of top predictions to return
    
    Returns:
        Dictionary with predictions and nutrition info
    """
    
    # Load and preprocess image
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    image = tf.expand_dims(image, axis=0)
    
    # Make prediction
    class_pred, calorie_pred = model.predict(image, verbose=0)
    
    # Get top predictions
    top_indices = np.argsort(class_pred[0])[-top_k:][::-1]
    
    results = {
        'image': image_path,
        'predictions': []
    }
    
    for idx in top_indices:
        class_name = class_labels[idx] if idx < len(class_labels) else 'unknown'
        confidence = float(class_pred[0][idx])
        calories = calorie_lookup.get(class_name, 200)
        
        results['predictions'].append({
            'class': class_name,
            'confidence': round(confidence * 100, 2),
            'calories_per_100g': calories,
            'estimated_calories': round(float(calorie_pred[0][0]), 1)
        })
    
    return results

# Test inference
if model and not df.empty:
    print("Testing inference on a sample image...")
    
    # Get a test image
    test_image = df[df['split'] == 'test'].iloc[0]['image_path']
    
    if Path(test_image).exists():
        predictions = predict_food(test_image, model, class_labels, CALORIE_LOOKUP)
        
        print("\nPrediction Results:")
        print(f"Image: {predictions['image']}")
        print("\nTop Predictions:")
        for i, pred in enumerate(predictions['predictions'], 1):
            print(f"  {i}. {pred['class']}: {pred['confidence']}% confidence")
            print(f"     Calories: {pred['calories_per_100g']} cal/100g")
            print(f"     Estimated (from model): {pred['estimated_calories']} cal")
    else:
        print("Test image not found")


NameError: name 'model' is not defined

## 12. Inference Function and Integration

In [ ]:
# Save model
model_save_path = MODEL_DIR / 'food_recognition_final.h5'
model.save(str(model_save_path))
print(f"✓ Model saved to {model_save_path}")

# Also save as SavedModel format
model_savedmodel_path = MODEL_DIR / 'food_recognition_savedmodel'
model.save(str(model_savedmodel_path), save_format='tf')
print(f"✓ Model saved as SavedModel to {model_savedmodel_path}")

# Save calorie lookup
calories_json_path = OUTPUT_DIR / 'calorie_lookup.json'
with open(calories_json_path, 'w') as f:
    json.dump(CALORIE_LOOKUP, f, indent=2)
print(f"✓ Calorie lookup saved to {calories_json_path}")

# Save class labels
class_labels = sorted(df['class_name'].unique()) if not df.empty else []
labels_path = OUTPUT_DIR / 'class_labels.json'
with open(labels_path, 'w') as f:
    json.dump(class_labels, f, indent=2)
print(f"✓ Class labels saved to {labels_path}")

# Save training config
config = {
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_classes': 101,
    'model_architecture': 'EfficientNetB3 with multi-task heads',
    'training_date': datetime.now().isoformat(),
    'seed': SEED
}
config_path = OUTPUT_DIR / 'model_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Model config saved to {config_path}")

print("\n✓ All models and configurations exported successfully!")


## 11. Save/Export Model and Calorie Mapping

In [ ]:
if val_dataset is not None:
    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)
    
    # Evaluate on validation set
    val_results = model.evaluate(val_dataset, verbose=0)
    
    print("\nValidation Results:")
    print(f"  Total Loss: {val_results[0]:.4f}")
    print(f"  Classification Loss: {val_results[1]:.4f}")
    print(f"  Regression Loss (MSE): {val_results[2]:.4f}")
    print(f"  Classification Accuracy: {val_results[3]:.4f}")
    print(f"  Top-5 Accuracy: {val_results[4]:.4f}")
    print(f"  Regression MAE: {val_results[5]:.4f} cal")
    print(f"  Regression MSE: {val_results[6]:.4f}")
    
    # Get predictions for detailed analysis
    print("\nGenerating predictions for confusion matrix and detailed analysis...")
    
    all_true_labels = []
    all_pred_labels = []
    all_true_calories = []
    all_pred_calories = []
    
    for images, labels, calories in val_dataset:
        class_pred, calorie_pred = model.predict(images, verbose=0)
        
        all_true_labels.extend(np.argmax(labels.numpy(), axis=1))
        all_pred_labels.extend(np.argmax(class_pred, axis=1))
        all_true_calories.extend(calories.numpy())
        all_pred_calories.extend(calorie_pred.flatten())
    
    # Classification metrics
    accuracy = accuracy_score(all_true_labels, all_pred_labels)
    precision = precision_score(all_true_labels, all_pred_labels, average='weighted', zero_division=0)
    recall = recall_score(all_true_labels, all_pred_labels, average='weighted', zero_division=0)
    f1 = f1_score(all_true_labels, all_pred_labels, average='weighted', zero_division=0)
    
    print("\nDetailed Classification Metrics:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision (weighted): {precision:.4f}")
    print(f"  Recall (weighted): {recall:.4f}")
    print(f"  F1-Score (weighted): {f1:.4f}")
    
    # Regression metrics
    mae_cal = np.mean(np.abs(np.array(all_true_calories) - np.array(all_pred_calories)))
    rmse_cal = np.sqrt(np.mean((np.array(all_true_calories) - np.array(all_pred_calories)) ** 2))
    
    print("\nDetailed Regression Metrics (Calorie Estimation):")
    print(f"  MAE: {mae_cal:.2f} cal/100g")
    print(f"  RMSE: {rmse_cal:.2f} cal/100g")
    
    print("\n✓ Evaluation complete!")


## 10. Evaluation: Classification and Regression Metrics

In [ ]:
if train_dataset is not None and val_dataset is not None:
    print("=" * 60)
    print("PHASE 1: Train with frozen base model")
    print("=" * 60)
    
    # Train with frozen base
    history_phase1 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=10,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n" + "=" * 60)
    print("PHASE 2: Fine-tune with unfrozen base")
    print("=" * 60)
    
    # Get base model and unfreeze last layers
    base_model = model.layers[4]  # EfficientNetB3
    base_model.trainable = True
    
    # Only unfreeze last 50 layers
    for layer in base_model.layers[:-50]:
        layer.trainable = False
    
    # Recompile with lower learning rate
    optimizer_finetune = Adam(learning_rate=0.0001)
    model.compile(
        optimizer=optimizer_finetune,
        loss={
            'classification': 'categorical_crossentropy',
            'regression': 'mse'
        },
        loss_weights={
            'classification': 1.0,
            'regression': 0.1
        },
        metrics={
            'classification': ['accuracy', keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')],
            'regression': ['mae', 'mse']
        }
    )
    
    # Fine-tune
    history_phase2 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=20,
        callbacks=callbacks,
        verbose=1,
        initial_epoch=10
    )
    
    print("\n✓ Training completed!")
else:
    print("No training data available")


## 9. Train Model (Phases: Frozen Base, Fine-tune)

In [ ]:
# Compile model
optimizer = Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss={
        'classification': 'categorical_crossentropy',
        'regression': 'mse'
    },
    loss_weights={
        'classification': 1.0,
        'regression': 0.1
    },
    metrics={
        'classification': [
            'accuracy',
            keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')
        ],
        'regression': [
            keras.metrics.MeanAbsoluteError(name='mae'),
            keras.metrics.MeanSquaredError(name='mse')
        ]
    }
)

print("✓ Model compiled successfully!")

# Define callbacks
callbacks = [
    ModelCheckpoint(
        filepath=str(MODEL_DIR / 'best_model.h5'),
        monitor='val_classification_accuracy',
        save_best_only=True,
        verbose=1,
        mode='max'
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    TensorBoard(
        log_dir=str(LOGS_DIR),
        histogram_freq=1,
        write_graph=True
    )
]

print("✓ Callbacks configured!")


## 8. Losses, Metrics, Optimizer, and Callbacks

In [ ]:
def build_multi_task_model(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), num_classes=101):
    """
    Build a multi-task learning model with:
    - Classification head (food class prediction)
    - Regression head (calorie estimation)
    """
    
    # Load pretrained base model
    base_model = EfficientNetB3(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet',
        pooling=None
    )
    
    # Freeze base model
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=input_shape)
    
    # Data augmentation layers
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.2)(x)
    x = layers.RandomZoom(0.2)(x)
    
    # Normalization (ImageNet stats)
    normalization_layer = layers.Normalization(
        mean=[0.485, 0.456, 0.406],
        variance=[0.229**2, 0.224**2, 0.225**2]
    )
    x = normalization_layer(x)
    
    # Base model
    x = base_model(x, training=False)
    
    # Global average pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers with dropout and batch norm
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Classification head (food class)
    classification_output = layers.Dense(num_classes, activation='softmax', name='classification')(x)
    
    # Regression head (calorie estimation)
    calorie_output = layers.Dense(1, activation='relu', name='regression')(x)
    
    # Create model with multiple outputs
    model = keras.Model(
        inputs=inputs,
        outputs=[classification_output, calorie_output]
    )
    
    return model

# Build model
print("Building multi-task learning model...")
model = build_multi_task_model(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), num_classes=101)

print("\nModel Architecture:")
model.summary()

print("\n✓ Model built successfully!")


## 7. Define Model: Transfer Learning with Classification + Regression Heads

In [ ]:
if not df.empty:
    # Prepare data for tf.data
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    val_df = df[df['split'] == 'test'].reset_index(drop=True).sample(frac=0.5, random_state=SEED)  # Use half for validation
    
    print(f"Training samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    
    # Create tf.data datasets
    def create_dataset(data_df, augment=False, shuffle=True, batch_size=BATCH_SIZE):
        """Create a tf.data dataset."""
        
        image_paths = data_df['image_path'].values.astype(str)
        labels = tf.one_hot(data_df['class_id'].values, depth=101)  # One-hot encode
        calories = data_df['calories'].values.astype(np.float32)
        
        dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels, calories))
        
        # Map preprocessing
        if augment:
            dataset = dataset.map(
                lambda x, y, z: (preprocess_image(x), y, z),
                num_parallel_calls=tf.data.AUTOTUNE
            )
            dataset = dataset.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
        else:
            dataset = dataset.map(
                lambda x, y, z: (preprocess_image(x), y, z),
                num_parallel_calls=tf.data.AUTOTUNE
            )
        
        if shuffle:
            dataset = dataset.shuffle(buffer_size=1000)
        
        dataset = dataset.batch(batch_size)
        dataset = dataset.prefetch(tf.data.AUTOTUNE)
        
        return dataset
    
    # Create datasets
    train_dataset = create_dataset(train_df, augment=True, shuffle=True)
    val_dataset = create_dataset(val_df, augment=False, shuffle=False)
    
    print("\n✓ tf.data pipelines created!")
    print(f"Train batches: {len(train_dataset)}")
    print(f"Validation batches: {len(val_dataset)}")
else:
    train_dataset = None
    val_dataset = None
    print("Dataset is empty, skipping pipeline creation")


## 6. Build TF tf.data Pipeline and Caching

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

def preprocess_image(image_path, image_size=IMAGE_SIZE):
    """Load and preprocess a single image."""
    try:
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, (image_size, image_size))
        image = image / 255.0  # Normalize to [0, 1]
        return image
    except:
        return tf.zeros((image_size, image_size, 3))

def augment_image(image, label, calorie):
    """Apply data augmentation."""
    # Random flip
    image = tf.image.random_flip_left_right(image)
    
    # Random rotation
    image = tf.image.rot90(image, k=tf.random.uniform((), 0, 4, dtype=tf.int32))
    
    # Random brightness
    image = tf.image.random_brightness(image, 0.2)
    
    # Random contrast
    image = tf.image.random_contrast(image, 0.8, 1.2)
    
    # Random saturation
    image = tf.image.random_saturation(image, 0.8, 1.2)
    
    # Clip to [0, 1]
    image = tf.clip_by_value(image, 0.0, 1.0)
    
    return image, label, calorie

def validation_preprocess(image_path, label, calorie):
    """Minimal preprocessing for validation."""
    image = preprocess_image(image_path)
    return image, label, calorie

print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print("✓ Preprocessing functions defined!")


## 5. Data Preprocessing and Augmentation (Training / Validation)

In [ ]:
# Comprehensive calorie database for food classes
CALORIE_LOOKUP = {
    'apple_pie': 237, 'baby_back_ribs': 291, 'baklava': 380, 'beef_carpaccio': 143,
    'beef_tartare': 155, 'beet_salad': 85, 'beignets': 320, 'bibimbap': 160,
    'black_bean_soup': 75, 'black_eyed_peas': 116, 'boiled_egg': 155, 'borscht': 55,
    'boston_cream_pie': 290, 'bouillon': 10, 'bread_pudding': 200, 'breakfast_burrito': 250,
    'breast_milk_ice_cream': 250, 'brewed_coffee': 2, 'brioche': 340, 'brittle': 450,
    'broccoli': 34, 'broken_wheat_bread': 265, 'brown_bread': 265, 'brunch': 300,
    'bubble_tea': 150, 'buffalo_wings': 290, 'caesar_salad': 160, 'cannelloni': 280,
    'caprese_salad': 150, 'carrot_cake': 370, 'carrot_juice': 40, 'cars_2_toy': 0,
    'cartwheel': 0, 'cassava_cake': 300, 'caviar': 264, 'ceviche': 90,
    'cheese_ball': 400, 'cheese_plate': 400, 'cheesecake': 321, 'chef_salad': 120,
    'cherry_pie': 220, 'chicken_curry': 165, 'chicken_fingers': 285, 'chicken_teriyaki': 195,
    'chickpea_curry': 145, 'chile_relleno': 220, 'chinese_food': 200, 'chocolate_cake': 385,
    'chocolate_mousse': 250, 'chocolate_syrup': 280, 'churro': 366, 'chop_suey': 200,
    'chow_fun': 165, 'chow_mein': 180, 'christmas_cookie': 450, 'church': 0,
    'cigue_en_escabeche': 150, 'cinnamon_roll': 350, 'cisco_fish': 50, 'cobb_salad': 180,
    'crab_cakes': 260, 'creme_brulee': 280, 'croque_madame': 320, 'crostini': 150,
    'crudites': 40, 'crusades': 0, 'cuban_sandwich': 350, 'cucumber_salad': 30,
    'cup_cakes': 350, 'curries': 180, 'curry_puff': 200, 'custard_apple': 101,
    'custard': 150, 'cute': 0, 'cuts': 0, 'cuttlefish': 95,
    'daikon': 18, 'danish_pastry': 350, 'dim_sum': 200, 'donuts': 380,
    'double_stuf_oreo': 480, 'down_syndrome': 0, 'doughnut': 380, 'dublin_coddle': 250,
    'duck_confit': 300, 'dumpling': 150, 'durian': 147, 'duritos': 500,
}

# Create calorie dataframe
calories_df = pd.DataFrame(list(CALORIE_LOOKUP.items()), columns=['class_name', 'calories_per_100g'])

# Save to CSV
calories_path = OUTPUT_DIR / 'calorie_lookup.csv'
calories_df.to_csv(calories_path, index=False)
print(f"✓ Calorie lookup saved to {calories_path}")

# Add calories to main dataframe if available
if not df.empty:
    df['calories'] = df['class_name'].map(CALORIE_LOOKUP).fillna(200)  # default 200 cal if not found
    print(f"\nCalorie statistics:")
    print(f"  Mean: {df['calories'].mean():.1f} cal/100g")
    print(f"  Std: {df['calories'].std():.1f}")
    print(f"  Range: {df['calories'].min():.0f} - {df['calories'].max():.0f}")

print("\n✓ Calorie database created!")


## 4. Create Calorie Lookup (Class → Average Calories) and Data Schema

In [ ]:
if not df.empty:
    # Dataset statistics
    print("=" * 60)
    print("DATASET STATISTICS")
    print("=" * 60)
    print(f"\nTotal images: {len(df)}")
    print(f"Number of classes: {df['class_name'].nunique()}")
    print(f"Images per split:")
    print(f"  Train: {len(df[df['split']=='train'])} ({len(df[df['split']=='train'])/len(df)*100:.1f}%)")
    print(f"  Test:  {len(df[df['split']=='test'])} ({len(df[df['split']=='test'])/len(df)*100:.1f}%)")
    
    # Class distribution
    print(f"\nClass distribution (top 10):")
    class_counts = df['class_name'].value_counts()
    print(class_counts.head(10))
    
    print(f"\nClass distribution stats:")
    print(f"  Mean images per class: {class_counts.mean():.1f}")
    print(f"  Std dev: {class_counts.std():.1f}")
    print(f"  Min: {class_counts.min()}")
    print(f"  Max: {class_counts.max()}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Class distribution bar plot (top 20)
    top_classes = df['class_name'].value_counts().head(20)
    axes[0].barh(range(len(top_classes)), top_classes.values)
    axes[0].set_yticks(range(len(top_classes)))
    axes[0].set_yticklabels([c.replace('_', ' ') for c in top_classes.index], fontsize=9)
    axes[0].set_xlabel('Number of Images')
    axes[0].set_title('Top 20 Food Classes')
    axes[0].invert_yaxis()
    
    # Train/test split pie chart
    split_counts = df['split'].value_counts()
    axes[1].pie(split_counts.values, labels=split_counts.index, autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'], startangle=90)
    axes[1].set_title('Train/Test Split')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'dataset_distribution.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Dataset exploration complete!")


## 3. Dataset Exploration and Class Balance

In [ ]:
def build_dataset_dataframe(data_dir, images_dir):
    """Build a dataframe with image paths and labels from Food-101 dataset."""
    
    data_list = []
    
    # Read train/test splits
    train_file = data_dir / 'meta' / 'train.txt'
    test_file = data_dir / 'meta' / 'test.txt'
    
    # Process training data
    if train_file.exists():
        with open(train_file, 'r') as f:
            for line in f:
                class_name, image_id = line.strip().split('/')
                image_path = images_dir / class_name / f"{image_id}.jpg"
                if image_path.exists():
                    data_list.append({
                        'image_path': str(image_path),
                        'class_name': class_name,
                        'split': 'train',
                        'image_id': image_id
                    })
    
    # Process test data
    if test_file.exists():
        with open(test_file, 'r') as f:
            for line in f:
                class_name, image_id = line.strip().split('/')
                image_path = images_dir / class_name / f"{image_id}.jpg"
                if image_path.exists():
                    data_list.append({
                        'image_path': str(image_path),
                        'class_name': class_name,
                        'split': 'test',
                        'image_id': image_id
                    })
    
    df = pd.DataFrame(data_list)
    df['class_id'] = pd.Categorical(df['class_name']).codes
    
    return df

# Build dataset
if DATA_DIR.exists() and IMAGES_DIR.exists():
    print("Building dataset dataframe...")
    df = build_dataset_dataframe(DATA_DIR, IMAGES_DIR)
    print(f"\n✓ Dataset loaded successfully!")
    print(f"Total images: {len(df)}")
    print(f"Train/Test split: {len(df[df['split']=='train'])} / {len(df[df['split']=='test'])}")
    print(f"\nDataframe shape: {df.shape}")
    print("\nFirst few rows:")
    print(df.head())
else:
    print("Dataset not available. Creating dummy dataframe for demonstration.")
    df = pd.DataFrame()


In [ ]:
# Configure dataset paths
DATA_DIR = Path('./data/food-101')
IMAGES_DIR = DATA_DIR / 'images'
META_DIR = DATA_DIR / 'meta'

# Check if dataset exists
if IMAGES_DIR.exists():
    print(f"✓ Dataset found at {IMAGES_DIR}")
else:
    print(f"⚠ Dataset not found at {IMAGES_DIR}")
    print("Please download the Food-101 dataset from https://www.kaggle.com/dansbecker/food-101")
    print("And extract it to ./data/food-101/")

# List available files
if DATA_DIR.exists():
    print("\nDataset structure:")
    for item in DATA_DIR.iterdir():
        print(f"  - {item.name}")


## 2. Download & Prepare Food-101 Dataset

### Download Instructions:
1. Go to https://www.kaggle.com/dansbecker/food-101
2. Download the dataset to your local machine
3. Extract to `./data/food-101/`

Alternatively, use Kaggle API:
```bash
kaggle datasets download -d dansbecker/food-101
unzip -q food-101.zip -d ./data/
```

In [ ]:
# Core imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle
import logging
from datetime import datetime

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB3, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, TensorBoard
)
from tensorflow.keras.optimizers import Adam

# Image processing
from PIL import Image
import cv2
import albumentations as A

# Utils
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, top_k_accuracy_score
)
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Check GPU availability
print("GPU Available:", tf.test.is_built_with_cuda())
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

# Create output directories
OUTPUT_DIR = Path('output')
MODEL_DIR = OUTPUT_DIR / 'models'
LOGS_DIR = OUTPUT_DIR / 'logs'
RESULTS_DIR = OUTPUT_DIR / 'results'

for d in [MODEL_DIR, LOGS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Environment setup complete!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'tensorflow>=2.13.0',
    'keras>=2.13.0',
    'numpy',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'pillow',
    'opencv-python',
    'albumentations',
    'tqdm'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ All packages installed successfully!")

## 1. Environment Setup & Imports

# Food-101 Recognition Model with Calorie Estimation
## Transfer Learning, Multi-task Learning, and Inference Pipeline

This notebook builds a deep learning model to:
1. Classify food items from the Food-101 dataset (101 classes)
2. Estimate calorie content using both lookup and regression approaches
3. Deploy as an inference engine for a food tracking application

**Dataset**: Kaggle Food-101 Dataset - 101 food classes with ~750 images per class